## Reformatting LISA datasets

The LISA datasets-- traffic lights and traffic signs-- are stored in a series of different folders. Within each folder is a csv file, where each row corresponds to one object. In order to train the YOLO model, I need three folders-- a train, validate, and test folder-- which each contain two folders: images and labels. For each image, I have a imageName.jpg file in the images folder-- the image itself-- and a imageName.txt file in the labels folder, with one row per object in that image.

This script converts from the LISA format into the YOLO format.

In [1]:
# Importing libraries
import os
import pandas as pd
import numpy as np
import math
import shutil

In [2]:
# Getting all training images
lisa_relative_path = "raw_images/LISA_traffic_lights"
day_folders_train = [f"{lisa_relative_path}/dayTrain/dayClip{i}" for i in [1, 5, 7, 9, 3, 11, 13]]
night_folders_train = [f"{lisa_relative_path}/nightTrain/nightClip{i}" for i in [1, 2, 3]]
folders_train = day_folders_train + night_folders_train

day_folders_valid = [f"{lisa_relative_path}/dayTrain/dayClip{i}" for i in [2, 8, 12]]
night_folders_valid = [f"{lisa_relative_path}/nightTrain/nightClip{i}" for i in [4]]
folders_valid = day_folders_valid + night_folders_valid

day_folders_test = [f"{lisa_relative_path}/dayTrain/dayClip{i}" for i in [4, 6, 10]]
night_folders_test = [f"{lisa_relative_path}/nightTrain/nightClip{i}" for i in [5]]
folders_test = day_folders_test + night_folders_test

names = ["train", "valid", "test"]

# Creating two folder sets: one with labels and images for day, one for night
save = True # Whether I save the images in the YOLO format
dest_dir = "YOLO_data/LISA_traffic_lights"
day_dir = os.path.join(dest_dir, "day")
night_dir = os.path.join(dest_dir, "night")
if not os.path.exists(dest_dir): os.mkdir(dest_dir)
if not os.path.exists(day_dir): os.mkdir(day_dir)
if not os.path.exists(night_dir): os.mkdir(night_dir)

for this_dir in [day_dir, night_dir]:
    for next_dir in names:
        next_path = os.path.join(this_dir, next_dir)
        if not os.path.exists(next_path): os.mkdir(next_path)

        img_dir = os.path.join(next_path, 'images')
        txt_dir = os.path.join(next_path, 'labels')
        if not os.path.exists(img_dir): os.mkdir(img_dir)
        if not os.path.exists(txt_dir): os.mkdir(txt_dir)

for f in folders_train: print(f)

raw_images/LISA_traffic_lights/dayTrain/dayClip1
raw_images/LISA_traffic_lights/dayTrain/dayClip5
raw_images/LISA_traffic_lights/dayTrain/dayClip7
raw_images/LISA_traffic_lights/dayTrain/dayClip9
raw_images/LISA_traffic_lights/dayTrain/dayClip3
raw_images/LISA_traffic_lights/dayTrain/dayClip11
raw_images/LISA_traffic_lights/dayTrain/dayClip13
raw_images/LISA_traffic_lights/nightTrain/nightClip1
raw_images/LISA_traffic_lights/nightTrain/nightClip2
raw_images/LISA_traffic_lights/nightTrain/nightClip3


In [3]:
# Copying images from into YOLO Folder

# Determining list of unique labels
unique_dict = {0: 'stop', 1: 'stopLeft', 2: 'go', 3: 'goLeft', 4: 'warning', 5: 'warningLeft'}
lookup_dict = {}
for key, value in unique_dict.items(): lookup_dict[value] = key
first_lookup_dict = {}

folder_list = [folders_train, folders_valid, folders_test]

for fnum in range(3):
    this_name = names[fnum]
    this_folder = folder_list[fnum]
    for x, f in enumerate(this_folder):
        print(x, f)
    
        # Loading the data frame for this folders' training sequence
        csv_path = os.path.join(lisa_relative_path, f"Annotations/Annotations/{f[31:]}/frameAnnotationsBOX.csv")
        col_names = ['file_name', 'label', 'xmin', 'ymin', 'xmax', 'ymax', 'origin', 'frame', 'origin2', 'frame2']
        label_df = pd.read_csv(csv_path, sep=';', header=0, names=col_names)
    
        # Cleaning up filenames
        if "day" in f: label_df['file_name'] = 'frames/' + label_df['file_name'].str[12:]
        else: label_df['file_name'] = 'frames/' + label_df['file_name'].str[14:]
        # display(label_df.head())
    
        # Determining list of unique files
        unique_files = label_df['file_name'].unique()
        n_unique_files = unique_files.shape[0]
        all_indices = np.linspace(0, n_unique_files-1, n_unique_files-1)
    
        # Compiling all data for .txt file for each image
        for i, (img_path, txt_df) in enumerate(label_df.groupby('file_name')):
            lines = list()
            for row in txt_df.itertuples():
                xcenter = int((row.xmin + row.xmax) / 2) / 1280
                if xcenter >= 1: print(row.xmin, row.xmax, xcenter)
                ycenter = int((row.ymin + row.ymax) / 2) / 960
                xwidth = (row.xmax - row.xmin) / 1280
                ywidth = (row.ymax - row.ymin) / 960
                line = f"{lookup_dict[row.label]} {xcenter} {ycenter} {xwidth} {ywidth}"
                lines.append(line)
                if float(line.split(' ')[1]) > 1.0:
                    print(line)
    
            # Determining if these files belong in the day or night folder
            this_mode = "night"
            if "day" in f: this_mode = "day"
    
            # Getting file paths
            old_img_path = f"{lisa_relative_path}/{this_mode}Train/{this_mode}Train/{f.rsplit('/', 1)[1]}/{img_path}"
            new_img_path = f"{dest_dir}/{this_mode}/{this_name}/images/{x:03d}_{i:04d}.jpg"
            new_txt_path = f"{dest_dir}/{this_mode}/{this_name}/labels/{x:03d}_{i:04d}.txt"
    
            first_lookup_dict[new_txt_path] = old_img_path
    
            # Saving files to new YOLO location
            if save:
                shutil.copy(old_img_path, new_img_path)
                with open(new_txt_path, 'w') as file:
                    for line in lines:
                        file.write(line + '\n')

0 raw_images/LISA_traffic_lights/dayTrain/dayClip1
1 raw_images/LISA_traffic_lights/dayTrain/dayClip5
2 raw_images/LISA_traffic_lights/dayTrain/dayClip7
3 raw_images/LISA_traffic_lights/dayTrain/dayClip9
4 raw_images/LISA_traffic_lights/dayTrain/dayClip3
5 raw_images/LISA_traffic_lights/dayTrain/dayClip11
6 raw_images/LISA_traffic_lights/dayTrain/dayClip13
7 raw_images/LISA_traffic_lights/nightTrain/nightClip1
8 raw_images/LISA_traffic_lights/nightTrain/nightClip2
9 raw_images/LISA_traffic_lights/nightTrain/nightClip3
0 raw_images/LISA_traffic_lights/dayTrain/dayClip2
1 raw_images/LISA_traffic_lights/dayTrain/dayClip8
2 raw_images/LISA_traffic_lights/dayTrain/dayClip12
3 raw_images/LISA_traffic_lights/nightTrain/nightClip4
0 raw_images/LISA_traffic_lights/dayTrain/dayClip4
1 raw_images/LISA_traffic_lights/dayTrain/dayClip6
2 raw_images/LISA_traffic_lights/dayTrain/dayClip10
3 raw_images/LISA_traffic_lights/nightTrain/nightClip5


The next step is to create more manageable, usable datasets from this large dataset. To start, I will make two datasets: One will have 2,000 images-- 60% training, 20% validation, and 20% testing. Each component will be 50% day photos, 50% night photos. The other dataset will have the same ratios but only 20 total images, taken from the first dataset. This second dataset will be used only for proof of concept runs-- i.e. I'll use it to quickly train a model to establish that all parts of a workflow work correctly, but the results of this model will never matter.

Note that I also need to, for every dataset, adjust to the number of classes in the output. Basically, once I grab the X images for a dataset and copy them in, I need to detremine how many different classes of objects are in the datasets, adjust the classes listed in my .txt files, and create a corresponding .yaml file

In [4]:
# Creates three folders called 'test', 'train', and 'valid'
# Each with a "images" and "label" folder inside of them
# Inside of this_dir
def create_dataset_skeleton(this_dir):
    # Creating main dataset folder
    if not os.path.exists(this_dir): os.mkdir(this_dir)

    # Creating test/train/valid folders
    test_dir = os.path.join(this_dir, 'test')
    train_dir = os.path.join(this_dir, 'train')
    valid_dir = os.path.join(this_dir, 'valid')
    if not os.path.exists(test_dir): os.mkdir(test_dir)
    if not os.path.exists(train_dir): os.mkdir(train_dir)
    if not os.path.exists(valid_dir): os.mkdir(valid_dir)

    # Create images/labels folders
    for next_dir in ["test", "train", "valid"]:
        img_dir = os.path.join(this_dir, next_dir, 'images')
        txt_dir = os.path.join(this_dir, next_dir, 'labels')
        if not os.path.exists(img_dir): os.mkdir(img_dir)
        if not os.path.exists(txt_dir): os.mkdir(txt_dir)

# Taken from https://github.com/ncallahanml/potential_vehicle_projects/blob/main/Examples/YOLOv8Train%26Deployment.ipynb
# this io could be handled with PyYAML
def write_yaml_config(class_dict, save_path, primary_path=None, train_path="train/", test_path="test/", valid_path="valid/"):
    yaml_content = f"""

path: {primary_path}  # dataset root dir
train: {train_path}  # train images (relative to 'path')
val: {valid_path}  # val images (relative to 'path')
test: {test_path}

names:"""
    for i in sorted(class_dict.keys()):
        yaml_content += f"\n  {i}: {class_dict[i]}"
        
    assert save_path.endswith('.yaml'), 'End file with .yaml extension'
    with open(save_path, 'w') as file:
        file.write(yaml_content)
    return

# 


In [10]:
import glob
import random

def train_dataset_function():
    train_data = []
    test_data = []
    val_data = []

    # Getting list of image filenames
    day_train_filenames = glob.glob(f"{dest_dir}/day/train/images/*")
    night_train_filenames = glob.glob(f"{dest_dir}/night/train/images/*")
    day_valid_filenames = glob.glob(f"{dest_dir}/day/valid/images/*")
    night_valid_filenames = glob.glob(f"{dest_dir}/night/valid/images/*")
    day_test_filenames = glob.glob(f"{dest_dir}/day/valid/images/*")
    night_test_filenames = glob.glob(f"{dest_dir}/night/valid/images/*")
    
    # Shuffling list of filenames (to randomly select files to use)
    random.seed(240304)
    random.shuffle(day_train_filenames)
    random.seed(240304)
    random.shuffle(night_train_filenames)
    random.seed(240304)
    random.shuffle(day_valid_filenames)
    random.seed(240304)
    random.shuffle(night_valid_filenames)
    random.seed(240304)
    random.shuffle(day_test_filenames)
    random.seed(240304)
    random.shuffle(night_test_filenames)

    dataset_size = 3000 # Total number of images in the dataset
    
    day_proportion, night_proportion = 1.0, 0 # Must add to 1
    train_proportion, val_proportion, test_proportion = 0.6, 0.2, 0.2 # Must add to 1
    
    # Determining which classes appear in the dataset
    unique_dict = {0: 'stop', 1: 'stopLeft', 2: 'go', 3: 'goLeft', 4: 'warning', 5: 'warningLeft'} # Hardcoded; all classes in big dataset
    
    # Copying files into new dataset
    day_range_boundaries = [(day_train_filenames, "day", "train", 0, dataset_size * day_proportion * train_proportion),
                            (day_valid_filenames, "day", "valid", 0, dataset_size * day_proportion * val_proportion),
                            (day_test_filenames, "day", "test", 0, dataset_size * day_proportion * test_proportion)]
    night_range_boundaries = [(night_train_filenames, "night", "train", 0, dataset_size * night_proportion * train_proportion),
                              (night_valid_filenames, "night", "valid", dataset_size * night_proportion * train_proportion, dataset_size * night_proportion * (train_proportion + val_proportion)),
                              (night_test_filenames, "night", "test", dataset_size * night_proportion * (train_proportion + val_proportion), dataset_size * night_proportion * (train_proportion + val_proportion + test_proportion))]
    
    second_lookup_dict = {}

    # Getting necessary data for all records
    for j in range(len(day_range_boundaries)):
        t = day_range_boundaries[j]
        for i in range(int(t[3]), int(t[4])):
            print("HERE", i)
            # print(len(t), len(t[0]))
            # File-level information
            this_record = {}
            this_record['file_name'] = t[0][i]
            this_record['height'] = 960
            this_record['width'] = 1280
            this_record['image_id'] = t[0][i]
            this_record['annotations'] = []

            # Record-level information
            old_txt_path = f"{t[0][i][:-20]}/labels/{t[0][i][-12:-4]}.txt"
            with open(old_txt_path, 'r') as file:
                lines = []
                for line in file:
                    this_annotations = {}
                    this_annotations['bbox_mode'] = 1 # Points in XYWH format
                    this_annotations['category_id'] = int(line.split(' ')[0])

                    # Getting bounding box values
                    dec_values = line.split(' ')
                    int_values = []
                    int_values.append(1280 * float(dec_values[1])) # X
                    int_values.append(960 * float(dec_values[2])) # Y
                    int_values.append(1280 * float(dec_values[3])) # W
                    int_values.append(960 * float(dec_values[4])) # H
                    int_values[0] -= 0.5 * int_values[2]
                    int_values[1] -= 0.5 * int_values[3]
                    this_annotations['bbox'] = int_values
                    this_record['annotations'].append(this_annotations)
            
            if j == 0: train_data.append(this_record)
            if j == 1: val_data.append(this_record)
            if j == 2: test_data.append(this_record)

    print("TRAIN DATA LEN", len(train_data))
    return train_data

def val_dataset_function():
    train_data = []
    test_data = []
    val_data = []

    # Getting list of image filenames
    day_train_filenames = glob.glob(f"{dest_dir}/day/train/images/*")
    night_train_filenames = glob.glob(f"{dest_dir}/night/train/images/*")
    day_valid_filenames = glob.glob(f"{dest_dir}/day/valid/images/*")
    night_valid_filenames = glob.glob(f"{dest_dir}/night/valid/images/*")
    day_test_filenames = glob.glob(f"{dest_dir}/day/valid/images/*")
    night_test_filenames = glob.glob(f"{dest_dir}/night/valid/images/*")
    
    # Shuffling list of filenames (to randomly select files to use)
    random.seed(240304)
    random.shuffle(day_train_filenames)
    random.seed(240304)
    random.shuffle(night_train_filenames)
    random.seed(240304)
    random.shuffle(day_valid_filenames)
    random.seed(240304)
    random.shuffle(night_valid_filenames)
    random.seed(240304)
    random.shuffle(day_test_filenames)
    random.seed(240304)
    random.shuffle(night_test_filenames)

    dataset_size = 3000 # Total number of images in the dataset
    
    day_proportion, night_proportion = 1.0, 0 # Must add to 1
    train_proportion, val_proportion, test_proportion = 0.6, 0.2, 0.2 # Must add to 1
    
    # Determining which classes appear in the dataset
    unique_dict = {0: 'stop', 1: 'stopLeft', 2: 'go', 3: 'goLeft', 4: 'warning', 5: 'warningLeft'} # Hardcoded; all classes in big dataset
    
    # Copying files into new dataset
    day_range_boundaries = [(day_train_filenames, "day", "train", 0, dataset_size * day_proportion * train_proportion),
                            (day_valid_filenames, "day", "valid", 0, dataset_size * day_proportion * val_proportion),
                            (day_test_filenames, "day", "test", 0, dataset_size * day_proportion * test_proportion)]
    night_range_boundaries = [(night_train_filenames, "night", "train", 0, dataset_size * night_proportion * train_proportion),
                              (night_valid_filenames, "night", "valid", dataset_size * night_proportion * train_proportion, dataset_size * night_proportion * (train_proportion + val_proportion)),
                              (night_test_filenames, "night", "test", dataset_size * night_proportion * (train_proportion + val_proportion), dataset_size * night_proportion * (train_proportion + val_proportion + test_proportion))]
    
    second_lookup_dict = {}

    # Getting necessary data for all records
    for j in range(len(day_range_boundaries)):
        t = day_range_boundaries[j]
        for i in range(int(t[3]), int(t[4])):
            # File-level information
            this_record = {}
            this_record['file_name'] = t[0][i]
            this_record['height'] = 960
            this_record['width'] = 1280
            this_record['image_id'] = t[0][i]
            this_record['annotations'] = []

            # Record-level information
            old_txt_path = f"{t[0][i][:-20]}/labels/{t[0][i][-12:-4]}.txt"
            with open(old_txt_path, 'r') as file:
                lines = []
                for line in file:
                    this_annotations = {}
                    this_annotations['bbox_mode'] = 1 # Points in XYWH format
                    this_annotations['category_id'] = int(line.split(' ')[0])

                    # Getting bounding box values
                    dec_values = line.split(' ')
                    int_values = []
                    int_values.append(1280 * float(dec_values[1])) # X
                    int_values.append(960 * float(dec_values[2])) # Y
                    int_values.append(1280 * float(dec_values[3])) # W
                    int_values.append(960 * float(dec_values[4])) # H
                    int_values[0] -= 0.5 * int_values[2]
                    int_values[1] -= 0.5 * int_values[3]
                    this_annotations['bbox'] = int_values
                    this_record['annotations'].append(this_annotations)
            
            if j == 0: train_data.append(this_record)
            if j == 1: val_data.append(this_record)
            if j == 2: test_data.append(this_record)
            
    return val_data

def test_dataset_function():
    train_data = []
    test_data = []
    val_data = []

    # Getting list of image filenames
    day_train_filenames = glob.glob(f"{dest_dir}/day/train/images/*")
    night_train_filenames = glob.glob(f"{dest_dir}/night/train/images/*")
    day_valid_filenames = glob.glob(f"{dest_dir}/day/valid/images/*")
    night_valid_filenames = glob.glob(f"{dest_dir}/night/valid/images/*")
    day_test_filenames = glob.glob(f"{dest_dir}/day/valid/images/*")
    night_test_filenames = glob.glob(f"{dest_dir}/night/valid/images/*")
    
    # Shuffling list of filenames (to randomly select files to use)
    random.seed(240304)
    random.shuffle(day_train_filenames)
    random.seed(240304)
    random.shuffle(night_train_filenames)
    random.seed(240304)
    random.shuffle(day_valid_filenames)
    random.seed(240304)
    random.shuffle(night_valid_filenames)
    random.seed(240304)
    random.shuffle(day_test_filenames)
    random.seed(240304)
    random.shuffle(night_test_filenames)

    dataset_size = 3000 # Total number of images in the dataset
    
    day_proportion, night_proportion = 1.0, 0 # Must add to 1
    train_proportion, val_proportion, test_proportion = 0.6, 0.2, 0.2 # Must add to 1
    
    # Determining which classes appear in the dataset
    unique_dict = {0: 'stop', 1: 'stopLeft', 2: 'go', 3: 'goLeft', 4: 'warning', 5: 'warningLeft'} # Hardcoded; all classes in big dataset
  
    # Copying files into new dataset
    day_range_boundaries = [(day_train_filenames, "day", "train", 0, dataset_size * day_proportion * train_proportion),
                            (day_valid_filenames, "day", "valid", 0, dataset_size * day_proportion * val_proportion),
                            (day_test_filenames, "day", "test", 0, dataset_size * day_proportion * test_proportion)]
    night_range_boundaries = [(night_train_filenames, "night", "train", 0, dataset_size * night_proportion * train_proportion),
                              (night_valid_filenames, "night", "valid", dataset_size * night_proportion * train_proportion, dataset_size * night_proportion * (train_proportion + val_proportion)),
                              (night_test_filenames, "night", "test", dataset_size * night_proportion * (train_proportion + val_proportion), dataset_size * night_proportion * (train_proportion + val_proportion + test_proportion))]
    
    second_lookup_dict = {}

    # Getting necessary data for all records
    for j in range(len(day_range_boundaries)):
        t = day_range_boundaries[j]
        for i in range(int(t[3]), int(t[4])):
            # File-level information
            this_record = {}
            this_record['file_name'] = t[0][i]
            this_record['height'] = 960
            this_record['width'] = 1280
            this_record['image_id'] = t[0][i]
            this_record['annotations'] = []

            # Record-level information
            old_txt_path = f"{t[0][i][:-20]}/labels/{t[0][i][-12:-4]}.txt"
            with open(old_txt_path, 'r') as file:
                lines = []
                for line in file:
                    this_annotations = {}
                    this_annotations['bbox_mode'] = 1 # Points in XYWH format
                    this_annotations['category_id'] = int(line.split(' ')[0])

                    # Getting bounding box values
                    dec_values = line.split(' ')
                    int_values = []
                    int_values.append(1280 * float(dec_values[1])) # X
                    int_values.append(960 * float(dec_values[2])) # Y
                    int_values.append(1280 * float(dec_values[3])) # W
                    int_values.append(960 * float(dec_values[4])) # H
                    int_values[0] -= 0.5 * int_values[2]
                    int_values[1] -= 0.5 * int_values[3]
                    this_annotations['bbox'] = int_values
                    this_record['annotations'].append(this_annotations)
            
            if j == 0: train_data.append(this_record)
            if j == 1: val_data.append(this_record)
            if j == 2: test_data.append(this_record)
            
    return test_data

In [11]:
## Registering custom datasets
from detectron2.data import MetadataCatalog
from detectron2.data import DatasetCatalog

# Registering Datasets
DatasetCatalog.register("train_dataset_v06", train_dataset_function)
DatasetCatalog.register(  "val_dataset_v06", val_dataset_function)
DatasetCatalog.register( "test_dataset_v06", test_dataset_function)

# Setting metadata
MetadataCatalog.get("train_dataset_v06").thing_classes = ['stop', 'stopLeft', 'go', 'goLeft', 'warning', 'warningLeft']
MetadataCatalog.get(  "val_dataset_v06").thing_classes = ['stop', 'stopLeft', 'go', 'goLeft', 'warning', 'warningLeft']
MetadataCatalog.get( "test_dataset_v06").thing_classes = ['stop', 'stopLeft', 'go', 'goLeft', 'warning', 'warningLeft']

# later, to access the data:
train_data = DatasetCatalog.get("train_dataset_v06")
val_data =     DatasetCatalog.get("val_dataset_v06")
test_data =   DatasetCatalog.get("test_dataset_v06")


HERE 0
HERE 1
HERE 2
HERE 3
HERE 4
HERE 5
HERE 6
HERE 7
HERE 8
HERE 9
HERE 10
HERE 11
HERE 12
HERE 13
HERE 14
HERE 15
HERE 16
HERE 17
HERE 18
HERE 19
HERE 20
HERE 21
HERE 22
HERE 23
HERE 24
HERE 25
HERE 26
HERE 27
HERE 28
HERE 29
HERE 30
HERE 31
HERE 32
HERE 33
HERE 34
HERE 35
HERE 36
HERE 37
HERE 38
HERE 39
HERE 40
HERE 41
HERE 42
HERE 43
HERE 44
HERE 45
HERE 46
HERE 47
HERE 48
HERE 49
HERE 50
HERE 51
HERE 52
HERE 53
HERE 54
HERE 55
HERE 56
HERE 57
HERE 58
HERE 59
HERE 60
HERE 61
HERE 62
HERE 63
HERE 64
HERE 65
HERE 66
HERE 67
HERE 68
HERE 69
HERE 70
HERE 71
HERE 72
HERE 73
HERE 74
HERE 75
HERE 76
HERE 77
HERE 78
HERE 79
HERE 80
HERE 81
HERE 82
HERE 83
HERE 84
HERE 85
HERE 86
HERE 87
HERE 88
HERE 89
HERE 90
HERE 91
HERE 92
HERE 93
HERE 94
HERE 95
HERE 96
HERE 97
HERE 98
HERE 99
HERE 100
HERE 101
HERE 102
HERE 103
HERE 104
HERE 105
HERE 106
HERE 107
HERE 108
HERE 109
HERE 110
HERE 111
HERE 112
HERE 113
HERE 114
HERE 115
HERE 116
HERE 117
HERE 118
HERE 119
HERE 120
HERE 121
HERE 122
HER

In [12]:
# Building custom Trainer
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import DatasetEvaluator, COCOEvaluator

class MyTrainer(DefaultTrainer):

  @classmethod
  def build_evaluator(cls, cfg, dataset_name, output_folder=None):

    if output_folder is None:
        os.makedirs("yolo_eval", exist_ok=True)
        output_folder = "yolo_eval"

    return COCOEvaluator(dataset_name, cfg, False, output_folder)

In [13]:
# Training model on custom datasets

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2.data.catalog import DatasetCatalog

from detectron2.engine import DefaultTrainer

"""
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("train_dataset_v03",)
cfg.DATASETS.TEST = ("val_dataset_v03",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 2  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.SOLVER.MAX_ITER = 300    # 300 iterations seems good enough for this toy dataset; you will need to train longer for a practical dataset
cfg.SOLVER.STEPS = []        # do not decay learning rate
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # The "RoIHead batch size". 128 is faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 6  # only has one class (ballon). (see https://detectron2.readthedocs.io/tutorials/datasets.html#update-the-config-for-new-datasets)
# NOTE: this config means the number of classes, but a few popular unofficial tutorials incorrect uses num_classes+1 here.
"""

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("train_dataset_v06",)
cfg.DATASETS.TEST = ("val_dataset_v06",)
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.SOLVER.MAX_ITER = 300    # 300 iterations seems good enough for this toy dataset; you will need to train longer for a practical dataset
cfg.SOLVER.STEPS = []        # do not decay learning rate
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 64   # faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 6  # only has one class (ballon). 

"""
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("train_dataset_v05",)
cfg.DATASETS.TEST = ("val_dataset_v05",)

cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.001


cfg.SOLVER.WARMUP_ITERS = 1000
cfg.SOLVER.MAX_ITER = 1500 #adjust up if val mAP is still rising, adjust down if overfit
cfg.SOLVER.STEPS = (1000, 100)
cfg.SOLVER.GAMMA = 0.05

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 64
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 6

cfg.TEST.EVAL_PERIOD = 500
"""

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = MyTrainer(cfg) 
trainer.resume_or_load(resume=False)
trainer.train()

[03/04 11:15:16 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (7, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (7,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (24, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (24,) in the model! You might want to double check if this is expected.
Some model parameters or buffers are not found in the checkpoint:
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, 

[03/04 11:15:17 d2.engine.train_loop]: Starting training from iteration 0
[03/04 11:15:33 d2.utils.events]:  eta: 0:03:50  iter: 19  total_loss: 2.889  loss_cls: 1.938  loss_box_reg: 0.856  loss_rpn_cls: 0.04846  loss_rpn_loc: 0.03342    time: 0.7805  last_time: 0.8660  data_time: 0.0141  last_data_time: 0.0035   lr: 1.6068e-05  max_mem: 6160M
[03/04 11:15:48 d2.utils.events]:  eta: 0:03:24  iter: 39  total_loss: 2.694  loss_cls: 1.725  loss_box_reg: 0.8482  loss_rpn_cls: 0.04591  loss_rpn_loc: 0.05353    time: 0.7636  last_time: 0.8640  data_time: 0.0036  last_data_time: 0.0038   lr: 3.2718e-05  max_mem: 6160M
[03/04 11:16:03 d2.utils.events]:  eta: 0:03:08  iter: 59  total_loss: 2.222  loss_cls: 1.307  loss_box_reg: 0.8193  loss_rpn_cls: 0.05579  loss_rpn_loc: 0.03599    time: 0.7694  last_time: 0.6525  data_time: 0.0037  last_data_time: 0.0039   lr: 4.9367e-05  max_mem: 6160M
[03/04 11:16:19 d2.utils.events]:  eta: 0:02:52  iter: 79  total_loss: 1.85  loss_cls: 0.8969  loss_box_reg:

In [25]:
# For debugging
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1" 

In [14]:
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.85
predictor = DefaultPredictor(cfg)
evaluator = COCOEvaluator("val_dataset_v06", cfg, False)
val_loader = build_detection_test_loader(cfg, "val_dataset_v06")
inference_on_dataset(trainer.model, val_loader, evaluator)

trainer.test(evaluator, cfg, False) #dataset_name, cfg, False, output_folder))

[03/04 11:21:40 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from ./output/model_final.pth ...
WARNING [03/04 11:21:40 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
[03/04 11:21:40 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[03/04 11:21:40 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[03/04 11:21:40 d2.data.common]: Serializing 600 elements to byte tensors and concatenating them all ...
[03/04 11:21:40 d2.data.common]: Serialized dataset takes 0.16 MiB
[03/04 11:21:40 d2.evaluation.evaluator]: Start inference on 600 batches
[03/04 11:21:42 d2.evaluation.evaluator]: Inference done 11/600. Dataloading: 0.0010 s/iter. Inference: 0.1192 s/iter. Eval: 0.0003 s/iter. Total: 0.1204 s/iter. ETA=0:0

AttributeError: 'COCOEvaluator' object has no attribute 'DATASETS'

In [20]:
import cv2

cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.DATASETS.TEST = ("my_dataset_test", )
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7   # set the testing threshold for this model
predictor = DefaultPredictor(cfg)
test_metadata = MetadataCatalog.get("test_dataset_v06")

from detectron2.utils.visualizer import ColorMode
import glob

for d in random.sample(test_data, 10):
  im = cv2.imread(d["file_name"])
  outputs = predictor(im)
  v = Visualizer(im[:, :, ::-1],
                metadata=test_metadata, 
                scale=0.8
                 )
  out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
  cv2.imshow("Predictions", out.get_image()[:, :, ::-1])
  cv2.waitKey(0)
  cv2.destroyAllWindows()

[03/04 11:27:01 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from ./output/model_final.pth ...


In [24]:
print(outputs)
print(outputs['instances'])

{'instances': Instances(num_instances=1, image_height=960, image_width=1280, fields=[pred_boxes: Boxes(tensor([[788.5284, 183.3831, 845.5861, 252.5470]], device='cuda:0')), scores: tensor([0.8458], device='cuda:0'), pred_classes: tensor([1], device='cuda:0')])}


AttributeError: Cannot find field 'fields' in the given Instances!

In [25]:
import glob
pred_images = glob.glob("partial_day_sequence_1/*")
for i in pred_images:
    print(i)

partial_day_sequence_1/daySequence1--03586.jpg
partial_day_sequence_1/daySequence1--03498.jpg
partial_day_sequence_1/daySequence1--03869.jpg
partial_day_sequence_1/daySequence1--03549.jpg
partial_day_sequence_1/daySequence1--03813.jpg
partial_day_sequence_1/daySequence1--03915.jpg
partial_day_sequence_1/daySequence1--03673.jpg
partial_day_sequence_1/daySequence1--03236.jpg
partial_day_sequence_1/daySequence1--03942.jpg
partial_day_sequence_1/daySequence1--03986.jpg
partial_day_sequence_1/daySequence1--03732.jpg
partial_day_sequence_1/daySequence1--03966.jpg
partial_day_sequence_1/daySequence1--03964.jpg
partial_day_sequence_1/daySequence1--03718.jpg
partial_day_sequence_1/daySequence1--03463.jpg
partial_day_sequence_1/daySequence1--04011.jpg
partial_day_sequence_1/daySequence1--03565.jpg
partial_day_sequence_1/daySequence1--03753.jpg
partial_day_sequence_1/daySequence1--03247.jpg
partial_day_sequence_1/daySequence1--03700.jpg
partial_day_sequence_1/daySequence1--03499.jpg
partial_day_s

In [33]:
""" PREDICTING ON CONTINUOUS SEQUENCE OF IMAGES """
import glob
import os
import time
# os.mkdir("detectron2DayTestSequenceNew")
pred_images = glob.glob("dayTestSequenceNewUnlabeled/*")
# Making prediction
start_time = time.perf_counter()
for i in pred_images:
    img = cv2.imread(i)
    outputs = predictor(img)
    v = Visualizer(img[:, :, ::-1],
                metadata=test_metadata, 
                scale=0.8
                 )
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    cv2.imwrite(f"detectron2DayTestSequenceNew/{i.split('/')[1]}", out.get_image()[:, :, ::-1])
    print(f"detectron2DayTestSequenceNew/{i.split('/')[1]}")
end_time = time.perf_counter()
print(f"Predict Time: {end_time - start_time} s.")

"""
# Moving all prediction images into one folder
all_folders = glob.glob(f"dayTestSequenceNew/*")
for f in all_folders:
    these_files = glob.glob(f"{f}/*")
    if len(these_files) > 0:
        # print(these_files[0].split('/')[-1])
            
        shutil.copy(these_files[0], f"dayTestSequenceNew/{these_files[0].split('/')[-1]}") # Copying over file
        os.remove(these_files[0])
        os.rmdir(f)
"""

detectron2DayTestSequenceNew/daySequence1--03586.jpg
detectron2DayTestSequenceNew/daySequence1--03498.jpg
detectron2DayTestSequenceNew/daySequence1--03869.jpg
detectron2DayTestSequenceNew/daySequence1--03549.jpg
detectron2DayTestSequenceNew/daySequence1--03813.jpg
detectron2DayTestSequenceNew/daySequence1--03915.jpg
detectron2DayTestSequenceNew/daySequence1--03673.jpg
detectron2DayTestSequenceNew/daySequence1--03236.jpg
detectron2DayTestSequenceNew/daySequence1--03942.jpg
detectron2DayTestSequenceNew/daySequence1--03986.jpg
detectron2DayTestSequenceNew/daySequence1--03732.jpg
detectron2DayTestSequenceNew/daySequence1--03966.jpg
detectron2DayTestSequenceNew/daySequence1--03964.jpg
detectron2DayTestSequenceNew/daySequence1--03718.jpg
detectron2DayTestSequenceNew/daySequence1--03463.jpg
detectron2DayTestSequenceNew/daySequence1--04011.jpg
detectron2DayTestSequenceNew/daySequence1--03565.jpg
detectron2DayTestSequenceNew/daySequence1--03753.jpg
detectron2DayTestSequenceNew/daySequence1--032

'\n# Moving all prediction images into one folder\nall_folders = glob.glob(f"dayTestSequenceNew/*")\nfor f in all_folders:\n    these_files = glob.glob(f"{f}/*")\n    if len(these_files) > 0:\n        # print(these_files[0].split(\'/\')[-1])\n            \n        shutil.copy(these_files[0], f"dayTestSequenceNew/{these_files[0].split(\'/\')[-1]}") # Copying over file\n        os.remove(these_files[0])\n        os.rmdir(f)\n'

In [34]:
## Creating predictions video
import cv2
import os
from natsort import natsorted  # Install using: pip install natsort

def images_to_video(image_folder, video_output, fps=16):
    # Ensure the image folder exists
    if not os.path.exists(image_folder):
        print(f"Image folder '{image_folder}' does not exist.")
        return

    # Get the list of image files and sort them by filename
    image_files = natsorted([f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.jpeg', '.png', '.gif'))])

    # Read the first image to get dimensions
    first_image = cv2.imread(os.path.join(image_folder, image_files[0]))
    height, width, layers = first_image.shape

    # Define the VideoWriter object
    video_writer = cv2.VideoWriter(video_output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Write each image to the video
    for image_file in image_files:
        image_path = os.path.join(image_folder, image_file)
        img = cv2.imread(image_path)
        video_writer.write(img)

    # Release the VideoWriter object
    video_writer.release()
    print(f"Video created: {video_output}")

# Specify the input image folder and output video file
image_folder = "detectron2DayTestSequenceNew"
video_output = "Detectron_Day_Prediction_SequenceNew.mp4"

# Call the function to create the video
images_to_video(image_folder, video_output)


Video created: Detectron_Day_Prediction_SequenceNew.mp4
